# AIGC 5005 · Module 05 · Part 6
## Imbalance, tuning and saving a model

Companion notebook for the matching part of the Module 05 learning content. Read each explanation, **predict** what the next cell will show, run it, and compare. Change values and run again; you cannot break anything.

**Kernel:** choose the `module05-sklearn` conda environment (top right in VS Code). To make it: `conda create -n module05-sklearn python=3.11 ipykernel`, then `conda activate module05-sklearn` and `pip install -r requirements.txt`. `churn_pipeline.py` and `customers.py` in this folder are shared modules. The tuning cell takes under a minute.

In [1]:
from churn_pipeline import CATEGORICAL, NUMERIC, build_pipeline, load_split

X, y, X_train, X_test, y_train, y_test = load_split()

## class_weight="balanced"
A mistake on the rare class now counts for more.

In [2]:
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score

balanced = build_pipeline(class_weight="balanced").fit(X_train, y_train)
predicted = balanced.predict(X_test)
print(confusion_matrix(y_test, predicted))
print(round(precision_score(y_test, predicted), 3), round(recall_score(y_test, predicted), 3),
      round(f1_score(y_test, predicted), 3))
print(round(roc_auc_score(y_test, balanced.predict_proba(X_test)[:, 1]), 3))

[[291 138]
 [ 17  54]]
0.281 0.761 0.411
0.781


Recall jumps from 0.056 to 0.761. Precision falls, because more stayers are flagged. The AUC is unchanged: the ranking is the same and only the decision rule moved.

A fairer account comes from cross-validation:

In [3]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_validate

folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
metrics = ["accuracy", "precision", "recall", "f1", "roc_auc"]
results = cross_validate(build_pipeline(class_weight="balanced"), X, y, cv=folds, scoring=metrics)
for m in metrics:
    scores = results["test_" + m]
    print(f"{m:10s} {scores.mean():.3f} +/- {scores.std():.3f}")
print("f1 per fold", results["test_f1"].round(3))

accuracy   0.681 +/- 0.026
precision  0.263 +/- 0.029
recall     0.695 +/- 0.062
f1         0.381 +/- 0.040
roc_auc    0.759 +/- 0.036
f1 per fold [0.328 0.406 0.339 0.408 0.426]


Notice the spread. A difference of 0.03 between two models on one split is within the noise.

## GridSearchCV
To reach a step's setting, write the step name, two underscores and the parameter name.

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocess = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), NUMERIC),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL),
])
forest = Pipeline([
    ("prep", preprocess),
    ("model", RandomForestClassifier(class_weight="balanced", random_state=42)),
])
grid = {
    "model__n_estimators": [100, 300],
    "model__max_depth": [3, 5, None],
    "model__min_samples_leaf": [1, 5],
}
search = GridSearchCV(forest, grid, scoring="f1", cv=folds, n_jobs=-1)
search.fit(X_train, y_train)
print(search.best_params_)
print(round(search.best_score_, 3))

{'model__max_depth': 5, 'model__min_samples_leaf': 1, 'model__n_estimators': 300}
0.399


In [5]:
results = pd.DataFrame(search.cv_results_)
results.sort_values("rank_test_score")[["params", "mean_test_score", "std_test_score", "rank_test_score"]].head(5)

,params,mean_test_score,std_test_score,rank_test_score
5,"{'model__max_depth': 5, 'model__min_samples_le...",0.399192,0.022042,1
4,"{'model__max_depth': 5, 'model__min_samples_le...",0.395473,0.014338,2
11,"{'model__max_depth': None, 'model__min_samples...",0.395259,0.042532,3
10,"{'model__max_depth': None, 'model__min_samples...",0.389505,0.048694,4
7,"{'model__max_depth': 5, 'model__min_samples_le...",0.388676,0.019144,5


Several settings are effectively tied. Now the test set, **used once**:

In [6]:
best = search.best_estimator_
print(round(f1_score(y_test, best.predict(X_test)), 3))
print(round(roc_auc_score(y_test, best.predict_proba(X_test)[:, 1]), 3))

0.362
0.751


The tuned forest does **worse** than the simple balanced logistic regression (F1 0.411, AUC 0.781). A more complex model is not automatically better, and one test set is one noisy measurement. Report both results; do not tune on the test set.

## Saving and loading
Save the **whole pipeline**. A joblib file is a pickle: only load files you trust, and note the scikit-learn version.

In [7]:
import joblib
import sklearn
from pathlib import Path

joblib.dump(balanced, "churn_model.joblib")
Path("sklearn_version.txt").write_text(sklearn.__version__ + "\n", encoding="utf-8")

loaded = joblib.load("churn_model.joblib")
print((loaded.predict(X_test) == balanced.predict(X_test)).all())
print("saved with scikit-learn", sklearn.__version__)

True
saved with scikit-learn 1.9.1


`churn_model.joblib` and `sklearn_version.txt` are created in this folder and ignored by Git.

## Try it
Run the grid search with `scoring="roc_auc"` instead of `"f1"`. Does the best setting change?